# **Кластеризация данных (Clustering Analysis)**

* __Цель кластеризации:__ Разбить совокупность пациентов на однородные группы (кластеры) на основе их медицинских показателей.
* __Задачи кластеризации:__ Применить алгоритмы обучения без учителя (Unsupervised Learning) для поиска скрытых паттернов в данных о пациентах.
* __Алгоритм использования:__
  1. __Метод K-средних (K-Means)__
  2. __Алгоритм DBSCAN (Пространственная кластеризация при наличии шума)__
  3. __Иерархическая кластеризация (Агломеративный подход)__

---

## **1. Подготовка данных к кластеризации**
* __Цель:__ Сформировать корректную, очищенную матрицу признаков для объективного обучения алгоритмов кластеризации "без учителя".
* __Задачи:__
  * Исключить из выборки целевую переменную, чтобы алгоритмы не имели к ней доступа при поиске скрытых паттернов.
  * Проверить итоговую матрицу на целостность (отсутствие пропущенных значений).
  * Оценить описательную статистику главных компонент (PCA) для подтверждения корректности масштабирования.

In [ ]:
# --- ИМПОРТ БИБЛИОТЕК И ПЕРВИЧНАЯ НАСТРОЙКА ---

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

# Установка параметра для воспроизводимости результатов
RANDOM_STATE = 42

# Настройка стиля графиков
sns.set_theme(
    style="whitegrid",
    palette="muted",
    rc={"figure.figsize": (10, 6), "axes.titlesize": 14},
)

In [ ]:
# --- 1. ЗАГРУЗКА ДАННЫХ ---

PROCESSED_DIR = Path("../data/processed")
INTERIM_DIR = Path("../data/interim")

data_pca = pd.read_csv(PROCESSED_DIR / "heart_disease_uci_pca.csv")
data_original = pd.read_csv(INTERIM_DIR / "heart_disease_uci_cleaned.csv")

In [ ]:
# --- 2. ПОДГОТОВКА МАТРИЦЫ ПРИЗНАКОВ ---

display(Markdown("#### **Формирование матрицы X**"))

# Отделение признаков (X) от целевой переменной / меток (y)
X_cluster = data_pca.drop(columns=["num"]).copy()
y_true = data_pca["num"].copy()

# Проверка отсутствия пропусков и корректности данных
missing_values = X_cluster.isna().sum().sum()

display(
    Markdown(
        f"* Матрица признаков `X` успешно сформирована.\n"
        f"* Размерность матрицы признаков `X`: **{X_cluster.shape[0]} строк и {X_cluster.shape[1]} главных компонент (PC)**.\n"
        f"* Пропущенных значений в данных PCA: **{missing_values}**."
    )
)

# Проверка распределения (PCA-компоненты должны иметь среднее ~0)
display(Markdown("#### **Описательная статистика признаков (X_cluster)**"))
display(X_cluster.describe().round(4))

## **2. Кластеризация методом K-Means**

### **2.1. Определение оптимального числа кластеров**

* __Цель:__ Определить математически обоснованное и практически интерпретируемое количество кластеров ($K$) для обучения модели K-Means.
* __Задачи:__
  1. Рассчитать внутрикластерную дисперсию (инерцию) для различных значений $K$ (Метод локтя).
  2. Вычислить коэффициент силуэта для оценки плотности и обособленности кластеров при различных $K$.
  3. Визуализировать метрики и выбрать оптимальное $K$ на основе консенсуса алгоритмов.

Главная проблема K-Means — выбор *K*. Методы для определения *K*:
* **Метод локтя (Elbow Method).** Оценивает сумму квадратов расстояний от точек до центроидов своих кластеров (инерцию). Оптимальным считается значение $K$, при котором на графике образуется характерный изгиб ("локоть"), где скорость уменьшения ошибки резко замедляется.
* **Коэффициент силуэта (Silhouette Score).** Измеряет плотность кластеров и расстояние между ними (принимает значения от -1 до 1). Чем ближе значение к 1, тем лучше объекты сгруппированы. Оптимальным считается $K$, при котором коэффициент достигает максимума.

In [ ]:
# --- 1. РАСЧЕТ МЕТРИК ---

# Списки для сохранения метрик
inertia_values = []
silhouette_scores = []

# Диапазон значений K
K_range = range(1, 11)
K_range_sil = range(2, 11)

display(
    Markdown("#### *Вычисление метрик (Инерция и Силуэт) для различных значений K...*")
)

for k in K_range:
    # Инициализация и обучение модели
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    kmeans.fit(X_cluster)

    # Сохранение инерции (Метод локтя)
    inertia_values.append(kmeans.inertia_)

    # Расчет коэффициента силуэта (K > 1)
    if k > 1:
        silhouette_avg = silhouette_score(X_cluster, kmeans.labels_)
        silhouette_scores.append(silhouette_avg)

# --- 2. ВИЗУАЛИЗАЦИЯ ---

plt.figure(figsize=(18, 6))

# График 1: Метод локтя (Elbow Method)
plt.subplot(1, 2, 1)
sns.lineplot(
    x=list(K_range),
    y=inertia_values,
    marker="o",
    color="teal",
    linewidth=2,
    markersize=8,
)
plt.title("Метод локтя (Elbow Method)", fontsize=14, pad=10)
plt.xlabel("Количество кластеров (K)", fontsize=12)
plt.ylabel("Инерция (Inertia)", fontsize=12)
plt.xticks(list(K_range))
plt.grid(True, linestyle="--", alpha=0.6)

# График 2: Коэффициент силуэта (Silhouette Score)
plt.subplot(1, 2, 2)
sns.lineplot(
    x=list(K_range_sil),
    y=silhouette_scores,
    marker="s",
    color="coral",
    linewidth=2,
    markersize=8,
)
plt.title("Коэффициент силуэта (Silhouette Score)", fontsize=14, pad=10)
plt.xlabel("Количество кластеров (K)", fontsize=12)
plt.ylabel("Средний коэффициент силуэта", fontsize=12)
plt.xticks(list(K_range_sil))
plt.grid(True, linestyle="--", alpha=0.6)

# Автоматическое нахождение и выделение максимума на графике силуэта
max_sil_index = np.argmax(silhouette_scores)
optimal_k_sil = K_range_sil[max_sil_index]
max_sil_value = silhouette_scores[max_sil_index]

plt.plot(optimal_k_sil, max_sil_value, marker="*", markersize=18, color="crimson")
plt.annotate(
    f" Максимум\n (K={optimal_k_sil})",
    xy=(optimal_k_sil, max_sil_value),
    xytext=(optimal_k_sil + 0.3, max_sil_value - 0.01),
    fontsize=12,
    color="darkred",
    weight="bold",
)

plt.tight_layout()
plt.show()

#### **Вывод по определению оптимального числа кластеров**

* **Анализ инерции (Метод локтя):** На левом графике наиболее выраженный излом ("локоть") наблюдается в диапазоне $K=2$ и $K=3$. Начиная с $K=3$, скорость снижения внутрикластерной дисперсии существенно падает, что говорит о нецелесообразности дальнейшего дробления данных.
* **Анализ плотности (Коэффициент силуэта):** Правый график демонстрирует однозначный глобальный максимум при $K=2$ (коэффициент силуэта $\approx 0.177$). Это математически подтверждает, что разбиение на две группы обеспечивает наилучшую сепарацию объектов в многомерном пространстве.
* **Итоговое обоснование:** Основываясь на консенсусе обеих метрик, для обучения алгоритма K-Means зафиксировано значение **$K=2$**. Данное математическое решение имеет сильный медицинский смысл в контексте данного набора данных: алгоритм обучения без учителя самостоятельно обнаружил скрытое бинарное разделение пациентов на две ключевые когорты — условную "группу нормы" и "группу риска".

### **2.2. Применение алгоритма K-Means**

* **Цель:** Выполнить сегментацию пациентов на основе выбранного оптимального значения $K$.
* **Задачи:**
  1. Инициализировать и обучить модель K-Means с заданными гиперпараметрами.
  2. Получить метки кластеров для каждого пациента.
  3. Визуализировать полученные кластеры в 2D-пространстве главных компонент (PCA) для наглядной оценки разделения.

In [ ]:
# --- ОБУЧЕНИЕ МОДЕЛИ ---

display(Markdown("#### **Обучение модели K-Means**"))

OPTIMAL_K = optimal_k_sil

# Инициализация и обучение модели
kmeans_final = KMeans(n_clusters=OPTIMAL_K, random_state=RANDOM_STATE, n_init=10)
kmeans_final.fit(X_cluster)

# Получение меток кластеров
cluster_labels = kmeans_final.labels_

# display(Markdown(f"* Модель успешно обучена. Получены метки для **{len(cluster_labels)}** объектов."))

display(
    Markdown(
        f"* Модель успешно обучена на **{OPTIMAL_K}** кластерах.\n"
        f"* Получены метки для **{len(cluster_labels)}** объектов."
    )
)

### **2.3. Визуализация и анализ**

* **Цель:** Наглядно оценить качество разделения и сформировать детальные профили полученных групп пациентов.
* **Задачи:**
  1. Визуализировать кластеры в 2D-проекции PCA с акцентом на центроиды.
  2. Рассчитать количественные характеристики сегментов (размер, доля).
  3. Проанализировать средние значения клинических показателей на исходных данных.
  4. Сформулировать выводы о медицинских профилях групп.

In [ ]:
# --- 1. ВИЗУАЛИЗАЦИЯ КЛАСТЕРОВ ---

display(Markdown("#### **Визуализация кластеров K-Means в 2D-пространстве**"))

# Создание копии X_cluster для графика и добавление меток
plot_df = X_cluster.copy()
plot_df["Cluster"] = cluster_labels.astype(str)

plt.figure(figsize=(12, 6))
sns.scatterplot(
    x="PC1",
    y="PC2",
    hue="Cluster",
    palette="Set1",
    data=plot_df,
    alpha=0.7,
    s=60,
    edgecolor="k",
)

# Добавление центроидов на график
centroids_pca = kmeans_final.cluster_centers_
plt.scatter(
    centroids_pca[:, 0],
    centroids_pca[:, 1],
    marker="X",
    s=100,
    c="black",
    label="Центроиды",
    edgecolor="white",
    linewidth=2,
)

plt.title("2D-проекция кластеров K-Means (PC1 vs PC2)", fontsize=14)
plt.xlabel("Главная компонента 1 (PC1)", fontsize=12)
plt.ylabel("Главная компонента 2 (PC2)", fontsize=12)
plt.legend(title="Кластер (K-Means)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# --- 2. ИНТЕРПРЕТАЦИЯ КЛАСТЕРОВ ---

display(Markdown("#### **Интерпретация кластеров**"))

# Создание копии оригинальных данных
data_profile = data_original.copy()

# Добавление полученных меток кластеров
data_profile["Cluster_KMeans"] = cluster_labels

# Расчет размеров кластеров
cluster_sizes = data_profile["Cluster_KMeans"].value_counts().sort_index()
cluster_percentages = (cluster_sizes / len(data_profile) * 100).round(1)

size_df = pd.DataFrame(
    {"Количество пациентов": cluster_sizes, "Доля (%)": cluster_percentages}
).reset_index()

display(Markdown("* **Количественное распределение:**"))
display(size_df)

# Исключение неинформативных столбцов перед расчетом средних значений признаков
cols_to_drop = ["id", "cluster"]
existing_drops = [col for col in cols_to_drop if col in data_profile.columns]

# Расчет средних значений признаков для каждого кластера
cluster_means = (
    data_profile.drop(columns=existing_drops)
    .groupby("Cluster_KMeans")
    .mean(numeric_only=True)
    .round(2)
)

display(Markdown("* **Средние значения медицинских показателей по кластерам:**"))
display(cluster_means.T)

#### **Выводы и описание сегментов (K-Means)**

**1. Визуализация (Разделимость кластеров):**
* На 2D-проекции отчетливо видно, что алгоритм K-Means успешно разделил пациентов на два крупных облака.
* Основная дисперсия и разделение происходят вдоль оси первой главной компоненты (PC1). 
* Несмотря на естественное для медицинских данных пересечение (overlap) в пограничной зоне, центроиды кластеров (черные кресты) находятся на значительном расстоянии друг от друга, что подтверждает математически качественную группировку.

**2. Интерпретация (Медицинские профили пациентов):**
Анализ средних значений исходных признаков позволяет составить четкие клинические портреты полученных групп:

* **Кластер 0 (Группа высокого риска / Пациенты с патологиями):**
  * **Размер:** 516 пациентов (56.2%).
  * **Профиль:** Пациенты старшей возрастной группы (средний возраст ~59 лет).
  * **Симптоматика:** Наблюдается повышенное артериальное давление в покое (`trestbps` = 134.5). Максимальная частота сердечных сокращений при нагрузке заметно снижена (`thalch` = 126). 
  * **Критические маркеры:** Более половины пациентов (54%) испытывают стенокардию, вызванную физической нагрузкой (`exang`), а показатель депрессии сегмента ST (`oldpeak`) критически завышен (1.22).
  * **Валидация (Диагноз):** Фактический средний показатель тяжести болезни (`num` = 1.45). *Хотя этот признак был скрыт от алгоритма*, K-Means абсолютно верно собрал в данный кластер людей с подтвержденными сердечно-сосудистыми заболеваниями.

* **Кластер 1 (Группа низкого риска / Условно здоровые):**
  * **Размер:** 402 пациента (43.8%).
  * **Профиль:** Более молодые пациенты (средний возраст ~47 лет).
  * **Симптоматика:** Давление находится в пределах нормы (`trestbps` = 125.7). Сердечно-сосудистая система отлично справляется с нагрузками: средний максимальный пульс высокий (`thalch` = 153).
  * **Критические маркеры:** Лишь 14% пациентов отмечают стенокардию при нагрузке, показатель `oldpeak` минимален (0.25).
  * **Валидация (Диагноз):** Показатель `num` (0.41) подтверждает, что модель "вслепую" выделила группу преимущественно здоровых пациентов.

## **3. Кластеризация методом DBSCAN**

### **3.1. Подбор параметров**

* **Цель:** Определить оптимальные гиперпараметры `eps` и `min_samples` для алгоритма пространственной кластеризации DBSCAN с учетом плотности медицинских данных.
* **Задачи:**
  1. Вычислить параметр `min_samples` на основе размерности признакового пространства.
  2. Построить график k-расстояний (k-distance graph) для оценки распределения плотности точек.
  3. Выбрать радиус окрестности `eps` на основе графика и специфики задачи.

Алгоритм DBSCAN (Density-Based Spatial Clustering of Applications with Noise) находит кластеры произвольной формы и автоматически определяет выбросы (шум). Для работы алгоритма требуются два ключевых параметра:
* `eps` — максимальное расстояние между двумя точками, чтобы они считались "соседями".
* `min_samples` — минимальное количество точек в окрестности радиуса `eps`, необходимое для образования ядра кластера.

In [ ]:
# --- 1. ОПРЕДЕЛЕНИЕ MIN_SAMPLES ---

display(Markdown("#### **Определение параметра eps**"))

# Рекомендация: n_features * 2
min_samples = X_cluster.shape[1] * 2
display(
    Markdown(
        f"* Выбрано значение `min_samples` (удвоенная размерность PCA): **{min_samples}**"
    )
)

# --- 2. МЕТОД K-РАССТОЯНИЙ ---

# Обучение модели NearestNeighbors для поиска расстояния до k-го соседа ((k = min_samples))
neighbors = NearestNeighbors(n_neighbors=min_samples)
neighbors_fit = neighbors.fit(X_cluster)

# Получение матрицы расстояний
distances, indices = neighbors_fit.kneighbors(X_cluster)

# Сортировка расстояния по убыванию
dist_k = np.sort(distances[:, min_samples - 1])[::-1]

In [ ]:
# --- 3. ПОСТРОЕНИЕ ГРАФИКА K-РАССТОЯНИЙ ДЛЯ ПОДБОРА EPS ---

# Визуализация графика k-расстояний
display(Markdown("#### **График k-расстояний (k-distance graph)**"))

plt.figure(figsize=(12, 6))
plt.plot(dist_k, color="teal", linewidth=2.5)
plt.title(f"График k-расстояний (k={min_samples}) для определения eps", fontsize=14)
plt.xlabel("Точки данных, отсортированные по убыванию расстояния", fontsize=12)
plt.ylabel(f"Расстояние до {min_samples}-го соседа (eps)", fontsize=12)

plt.grid(True, which="both", linestyle="--", alpha=0.7)
plt.minorticks_on()
plt.grid(True, which="minor", linestyle=":", alpha=0.4)

plt.tight_layout()
plt.show()

#### **Анализ графика и обоснование выбора параметров:**

* **Математический подход:** Визуально изгиб ("локоть") на графике k-расстояний начинается в диапазоне от **1.8 до 2.5**. По классической теории именно здесь следует искать оптимальный `eps`.
* **Специфика данных:** Однако, медицинские данные представляют собой непрерывный спектр с высокой плотностью в центре (переходная зона между здоровьем и патологией). Если выбрать классическое высокое значение (например, `eps = 2.3`), алгоритм DBSCAN сольет почти всех пациентов в один гигантский кластер из-за так называемого "цепного эффекта".
* **Бизнес-решение:** Чтобы разорвать эти связи и заставить алгоритм выделить именно самые плотные ядра (явно выраженных больных и явно здоровых пациентов), параметр `eps` был эмпирически занижен. 
* **Итог:** Для обучения модели фиксируются параметры **`eps = 1.4`** и **`min_samples = 16`**. Это неизбежно увеличит количество "шумовых" точек (выбросов), но позволит получить логически интерпретируемые сегменты.

### **3.2. Применение алгоритма DBSCAN**

* **Цель:** Реализовать плотностную кластеризацию и выделить наиболее устойчивые ядра групп пациентов, изолировав аномалии и "пограничные" случаи.
* **Задачи:**
  1. Обучить модель DBSCAN с выбранными параметрами `eps` = 1.4 и `min_samples`= 16.
  2. Рассчитать количественные характеристики: число найденных кластеров и объем отсеянного шума.
  3. Визуализировать результат, наглядно разделив структурные кластеры и шумовые точки.

In [ ]:
# --- 1. ОБУЧЕНИЕ МОДЕЛИ DBSCAN ---

display(Markdown("#### **Обучение модели DBSCAN**"))

# Ручной выбор параметров на основе графика k-расстояний
EPS_VALUE = 1.4
MIN_SAMPLES_VALUE = 16

# Инициализация и обучение модели
dbscan = DBSCAN(eps=EPS_VALUE, min_samples=MIN_SAMPLES_VALUE)
dbscan_labels = dbscan.fit_predict(X_cluster)

# Подсчет количества кластеров и шумовых точек
n_clusters_ = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise_ = list(dbscan_labels).count(-1)
noise_percent = (n_noise_ / len(X_cluster)) * 100

# --- 2. АНАЛИЗ РЕЗУЛЬТАТОВ ---

display(
    Markdown(
        f"* Заданные параметры: `eps` = **{EPS_VALUE}**, `min_samples` = **{MIN_SAMPLES_VALUE}**\n"
        f"* Количество найденных кластеров: **{n_clusters_}**\n"
        f"* Количество шумовых точек (выбросов): **{n_noise_}** ({noise_percent:.1f}% от всех данных)"
    )
)

In [ ]:
# --- 3. ВИЗУАЛИЗАЦИЯ ---

display(Markdown("#### **Визуализация кластеров DBSCAN в 2D-пространстве**"))

# Создание копии X_cluster для графика и добавление меток DBSCAN
plot_df_dbscan = X_cluster.copy()
plot_df_dbscan["Cluster_DBSCAN"] = dbscan_labels

plt.figure(figsize=(12, 6))

# Разделение данных для дифференцированной отрисовки
noise_data = plot_df_dbscan[plot_df_dbscan["Cluster_DBSCAN"] == -1]
core_data = plot_df_dbscan[plot_df_dbscan["Cluster_DBSCAN"] != -1]

# Отрисовка шумовых точек (серые крестики)
if not noise_data.empty:
    sns.scatterplot(
        x="PC1",
        y="PC2",
        data=noise_data,
        color="dimgrey",
        marker="x",
        s=30,
        label="Шум (-1)",
        alpha=0.7,
    )

# Отрисовка кластеров (яркие точки)
if not core_data.empty:
    core_data = core_data.copy()
    core_data["Cluster_DBSCAN"] = core_data["Cluster_DBSCAN"].astype(str)

    sns.scatterplot(
        x="PC1",
        y="PC2",
        hue="Cluster_DBSCAN",
        palette="tab10",
        data=core_data,
        s=60,
        alpha=0.7,
        edgecolor="k",
    )

plt.title(
    f"2D-проекция кластеров DBSCAN (eps={EPS_VALUE}, min_samples={MIN_SAMPLES_VALUE})",
    fontsize=14,
)
plt.xlabel("Главная компонента 1 (PC1)", fontsize=12)
plt.ylabel("Главная компонента 2 (PC2)", fontsize=12)
plt.legend(title="Кластер (DBSCAN)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

#### **Выводы и интерпретация результатов (DBSCAN)**

**1. Особенности плотностного разделения:**
* В отличие от K-Means, который принудительно разделил все точки жесткой границей, DBSCAN выделил только **самые плотные ядра** концентрации пациентов.
* Значительная часть данных (~40%) была отнесена к кластеру `-1` (шум). На визуализации отчетливо видно, что эти точки (серые маркеры) располагаются преимущественно в центре пространства, образуя "мост" между двумя основными цветными облаками.

**2. Медицинский смысл "шума" (Переходная зона):**
* В контексте кардиологических данных этот "шум" — это не системная ошибка или бракованные записи. Это пациенты, находящиеся в **переходной зоне (группа умеренного риска)**. 
* Их показатели уже начали отклоняться от идеальной нормы, но еще не достигли критических значений явной патологии. DBSCAN математически подтвердил, что переход от здоровья к болезни в наших данных — это непрерывный спектр, а не два изолированных состояния.

**3. Сравнительный анализ:**
* Если **K-Means** оптимален для агрессивной бинарной сегментации (охват 100% базы), то **DBSCAN** блестяще справляется с задачей выявления "идеальных" представителей каждой группы. 
* Алгоритм надежно изолировал явно здоровых пациентов и пациентов с тяжелой симптоматикой (ядра), оставив пограничные и сомнительные случаи (шум) для индивидуального или дополнительного врачебного изучения.